# Thesis Evaluation Notebook

Loads `.pkl` result files and produces all thesis plots.


In [ ]:
import os, sys, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.evaluate import f_max_new

In [ ]:
RESULTS_DIR = '../experiments/thesis_experiments/results_090526_thesis'  
FIGURES_DIR = '../experiments/thesis_experiments/figures' 
TABLES_DIR = '../experiments/thesis_experiments/tables'

MAX_CALLS = 300  

VARIANTS = [
     ('gibo_baseline', 'GIBO'),
     ('gibo_argmax_pwolfe', 'GIBO probWolfe'),
     ('gibo_argmax_detEI', 'GIBO wolfeEI'),
]

DIMENSIONS = [4, 8, 12, 16, 20, 24, 28, 32, 36]


os.makedirs(FIGURES_DIR, exist_ok=True)
print(f'Figures will be saved to: {FIGURES_DIR}')
os.makedirs(TABLES_DIR, exist_ok=True)
print(f'Tables will be saved to: {TABLES_DIR}')

## Block 1: Load pkl Files

In [ ]:
def load_variant_data(results_dir, name, dimensions):
    data = {}
    for k in [
        'calls', 'best_so_far', 'f_max',
        'inner_loop_samples', 'step_sizes', 'gradient_norms',
        'p_wolfe_values', 'wolfe_satisfied', 'armijo_ok', 'curvature_ok',
        'f_values']:# parameter positions per iteratio
        data[k] = {}
    
    for dim in dimensions:
        dim_dir = os.path.join(results_dir, name, f'dim_{dim}')
        if not os.path.isdir(dim_dir):
            continue
        
        pickle_files = sorted(f for f in os.listdir(dim_dir) if f.endswith('.pkl'))
        if not pickle_files:
            continue
        
        calls_dim, bsf_dim, fmax_dim = [], [], []
        ils_dim, ss_dim, gn_dim = [], [], []
        pw_dim, ws_dim, arm_dim, cur_dim = [], [], [], []
        fv_dim = []
        
        for fname in pickle_files:
            with open(os.path.join(dim_dir, fname), 'rb') as fh:
                r = pickle.load(fh)
            fmax = float(r['f_max'])
            regret = r['regret_per_eval']
            calls = r['calls_at_iteration']
            bsf = [fmax - rg for rg in regret]
            calls_dim.append(calls)
            bsf_dim.append([0.0] + bsf)
            fmax_dim.append(fmax)
            ils_dim.append(r.get('inner_loop_samples', []))
            ss_dim.append(r.get('step_sizes', []))
            gn_dim.append(r.get('gradient_norms', []))
            pw_dim.append(r.get('p_wolfe_values', []))
            ws_dim.append(r.get('wolfe_satisfied', []))
            arm_dim.append(r.get('armijo_ok', []))
            cur_dim.append(r.get('curvature_ok', []))
            fv_dim.append(r.get('f_values', []))
            
        data['calls'][dim] = calls_dim
        data['best_so_far'][dim] = bsf_dim
        data['f_max'][dim] = fmax_dim
        data['inner_loop_samples'][dim] = ils_dim
        data['step_sizes'][dim] = ss_dim
        data['gradient_norms'][dim] = gn_dim
        data['p_wolfe_values'][dim] = pw_dim
        data['wolfe_satisfied'][dim] = ws_dim
        data['armijo_ok'][dim] = arm_dim
        data['curvature_ok'][dim] = cur_dim
        data['f_values'][dim]= fv_dim
        print(f' dim={dim}: {len(pickle_files)} runs')
    return data


all_data = {}
for vname, vlabel in VARIANTS:
    print(f'Loading {vlabel} ({vname}) ...')
    all_data[vname] = load_variant_data(RESULTS_DIR, vname, DIMENSIONS)
    
available_dims = []
for d in DIMENSIONS:
    dimension_available = True
    
    for vname, _ in VARIANTS:
        if d not in all_data[vname]['calls']:
            dimension_available = False
            break
    if dimension_available:
        available_dims.append(d)
        
print(f'\nAvailable dims: {available_dims}')

## Block 2: Interpolation

In [ ]:
def interpolate_rewards(rewards_dict, calls_dict, max_calls):
    """
    Replacement for postprocessing_interpolation_rewards.
    Zero-Order-Hold Interpolation-->step function.
    """
    dimensions = list(rewards_dict.keys())
    n_runs = len(rewards_dict[dimensions[0]])
    new_rewards = torch.zeros(len(dimensions), n_runs, max_calls)
    for index_dim, dim in enumerate(dimensions):
        for index_run in range(n_runs):
            rewards = rewards_dict[dim][index_run]
            calls = calls_dict[dim][index_run]
            index_rewards = 0
            for call in range(max_calls):
                if index_rewards < len(calls) and call == calls[index_rewards]:
                    index_rewards += 1
                new_rewards[index_dim, index_run, call] = rewards[index_rewards]
    return new_rewards


interpolated = {}
for vname, vlabel in VARIANTS:
    data = all_data[vname]
    rewards_dict = {dim: data['best_so_far'][dim] for dim in available_dims}
    calls_dict = {dim: data['calls'][dim] for dim in available_dims}
    interpolated[vname] = interpolate_rewards(rewards_dict, calls_dict, MAX_CALLS)
    print(f'{vlabel}: {interpolated[vname].shape}')

first_vname = VARIANTS[0][0]
f_max_dict = {dim: all_data[first_vname]['f_max'][dim] for dim in available_dims}
f_max_corrected = f_max_new(f_max_dict, list(interpolated.values()))

names = [label for _, label in VARIANTS]
tensors = [interpolated[vname] for vname, _ in VARIANTS]

## Block 3: Regret vs Evaluations

In [ ]:
def plot_regret(f_max, rewards_list, names, available_dims, max_calls, show_std=True, path=None):
    markers = ['D', 'o', 's']
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    n = len(available_dims)
    n_cols = 3
    n_rows = int(np.ceil(n / n_cols))
    x = np.arange(max_calls)
    fig, axs = plt.subplots(n_rows, n_cols, sharex=True, sharey=False,
                             figsize=(5 * n_cols, 3.5 * n_rows))
    axs = np.array(axs).reshape(-1)

    for index_dim, dim in enumerate(available_dims):
        ax = axs[index_dim]
        fmax = np.array(f_max[dim]).reshape(-1, 1)
        ax.set_title(f'{dim}-dim', fontsize=12)
        ax.set_xlim([0, max_calls])
        if index_dim >= n_cols * (n_rows - 1):
            ax.set_xlabel('function evaluations', labelpad=5, fontsize=11)
        if index_dim % n_cols == 0:
            ax.set_ylabel('simple regret [log]', fontsize=11)

        for index_o, (rew, name) in enumerate(zip(rewards_list, names)):
            rew_np = rew[index_dim].numpy()
            err = (fmax - rew_np) / fmax
            err = np.clip(err, 1e-2, None) 
            mean = err.mean(0)
            std = err.std(0)
            ax.plot(x, mean, label=name,
                    marker=markers[index_o],
                    markevery=max(1, max_calls // 8),
                    markersize=4, linewidth=1.3,
                    color=colors[index_o], fillstyle='none')
            if show_std:
                ax.fill_between(x,np.clip(mean - std, 1e-2, None), mean + std,
                                alpha=0.12, color=colors[index_o])

        ax.set_yscale('log')
        ax.set_ylim([1e-2, None])
        ax.yaxis.grid(True, which='both', linewidth=0.3, alpha=0.5)
        ax.xaxis.grid(True, linewidth=0.3, alpha=0.4)

    for index in range(len(available_dims), n_rows * n_cols):
        axs[index].set_visible(False)

    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center',
               bbox_to_anchor=(0.5, 1.02), ncol=len(names),
               frameon=True, fontsize=12)
    fig.suptitle('Simple Regret vs. Function Evaluations', fontsize=15, y=1.06)
    plt.tight_layout()
    if path:
        plt.savefig(path, bbox_inches='tight')
        print(f'Saved: {path}')
    plt.show()

plot_regret(f_max_corrected, tensors, names, available_dims, MAX_CALLS, show_std=True,
            path=os.path.join(FIGURES_DIR, 'regret_vs_evals.pdf'))

## Block 4: Total Surrogate Steps over Dimensions

In [ ]:
def load_step_alpha(results_dir, variants, dimensions):
    step_totals = {}
    alpha_means = {}
    
    for vname, _ in variants:
        step_totals[vname] = {}
        alpha_means[vname] = {}
        
        for dim in dimensions:
            dim_dir = os.path.join(results_dir, vname, f'dim_{dim}')
            if not os.path.isdir(dim_dir):
                continue
            pickle_files = sorted(fn for fn in os.listdir(dim_dir) if fn.endswith('.pkl'))
            steps_list = []
            alpha_list = []
            
            for fname in pickle_files:
                with open(os.path.join(dim_dir, fname), 'rb') as fh:
                    r = pickle.load(fh)
                if 'baseline' in vname:
                    steps_list.append(len(r.get('inner_loop_samples', [])))
                    f_value = r.get('f_values', [])
                    displacement = []

                    for t in range(len(f_value) - 1):
                        displacement.append(
                           float(
                             np.linalg.norm(
                                np.array(f_value[t + 1]).flatten()
                                - np.array(f_value[t]).flatten()
                                 )
                            )
                        )
                    alpha_list.append(float(np.mean(displacement)) if displacement else float('nan'))
                else:
                    steps_list.append(sum(r.get('n_steps_per_iter', [])))
                    all_alpha = []
                    for row in r.get('alpha_history_per_iter', []):
                        for a in row:
                           all_alpha.append(a)
                           
                    alpha_list.append(float(np.mean(all_alpha)) if all_alpha else float('nan'))
            step_totals[vname][dim] = steps_list
            alpha_means[vname][dim] = alpha_list
    return step_totals, alpha_means

step_totals, alpha_means = load_step_alpha(RESULTS_DIR, VARIANTS, available_dims)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

medp = dict(linestyle='-', linewidth=1.2, color='black')
meanp = dict(marker='o', markerfacecolor='green', markersize=4, markeredgecolor='none')
flierp = dict(marker='o', markerfacecolor='gray', markersize=3, alpha=0.5, markeredgecolor='none')

n = len(VARIANTS)
if n <= 1:
    nrows, ncols, figsize = 1, 1, (6, 5)
    has_legend_cell = False
elif n == 2:
    nrows, ncols, figsize = 1, 2, (10, 4)
    has_legend_cell = False
else:
    nrows, ncols, figsize = 2, 2, (10, 8)
    has_legend_cell = True

fig, axs = plt.subplots(nrows, ncols, sharex=True, sharey=False, figsize=figsize)
axs_flat = np.array(axs).reshape(-1)

# different y-limits (extend or set None for auto)
ylim_per_index = [(0, 100), (100, 180 ), (100, 380)]

for index, (vname, vlabel) in enumerate(VARIANTS):
    ax = axs_flat[index]
    color = colors[index]

    data = []
    for dim in available_dims:
        run_steps = np.array(step_totals[vname].get(dim, []), dtype=float)
        data.append(run_steps)

    boxplot = ax.boxplot(
        data,
        positions=available_dims,
        widths=2.2,
        patch_artist=True,
        showfliers=True,
        showmeans=True,
        medianprops=medp,
        meanprops=meanp,
        flierprops=flierp,
        whiskerprops=dict(linewidth=0.8),
        boxprops=dict(linewidth=0.8),
    )

    for box in boxplot['boxes']:
       box.set_facecolor(color)
       box.set_alpha(0.5)

    ax.set_title(vlabel, fontsize=13)
    ax.set_xticks(available_dims)
    ax.tick_params(axis='x', labelrotation=45, labelsize=10)
    ax.tick_params(axis='y', labelsize=10)
    ax.yaxis.grid(True, linewidth=0.3, alpha=0.5)
    ax.set_xlabel('Dimension $d$', fontsize=11)
    if index % ncols == 0:
        ax.set_ylabel('Total steps per budget', fontsize=11)
    if index < len(ylim_per_index):
        ax.set_ylim(*ylim_per_index[index])
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=6))

# Hide the unused cells
for i in range(n, nrows * ncols):
    axs_flat[i].set_visible(False)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=9, label='mean'),
    Line2D([0], [0], color='black', lw=2, label='median'),
]
if has_legend_cell:
    fig.legend(handles=legend_handles, loc='lower right',
               bbox_to_anchor=(0.95, 0.12), frameon=True, fontsize=12,
               title='Legend', title_fontsize=11)
else:
    fig.legend(handles=legend_handles, loc='lower center',
               bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=True, fontsize=12,
               title='Legend', title_fontsize=11)

plt.suptitle('Total Steps per Budget over Dimensions', fontsize=15, y=1.01)
plt.tight_layout()

path = os.path.join(FIGURES_DIR, 'boxplot_steps.pdf')
plt.savefig(path, dpi=300, bbox_inches='tight')
print(f'Saved: {path}')
plt.show()


## Block 5: Boxplot: Mean Step Size α over Dimensions

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

medp = dict(linestyle='-', linewidth=1.2, color='black')
meanp = dict(marker='o', markerfacecolor='green', markersize=4, markeredgecolor='none')
flierp = dict(marker='o', markerfacecolor='gray', markersize=3, alpha=0.5, markeredgecolor='none')

n = len(VARIANTS)
if n <= 1:
    nrows, ncols, figsize = 1, 1, (6, 5)
    has_legend_cell = False
elif n == 2:
    nrows, ncols, figsize = 1, 2, (10, 4)
    has_legend_cell = False
else:
    nrows, ncols, figsize = 2, 2, (10, 8)
    has_legend_cell = True

fig, axs = plt.subplots(nrows, ncols, sharex=True, sharey=True, figsize=figsize)
axs_flat = np.array(axs).reshape(-1)

for index, (vname, vlabel) in enumerate(VARIANTS):
    ax = axs_flat[index]
    color = colors[index]

    data = []
    for dim in available_dims:
        run_alphas = [v for v in alpha_means[vname].get(dim, []) if not np.isnan(v)]
        data.append(np.array(run_alphas, dtype=float))

    boxplot = ax.boxplot(
        data,
        positions=available_dims,
        widths=2.2,
        patch_artist=True,
        showfliers=True,
        showmeans=True,
        medianprops=medp,
        meanprops=meanp,
        flierprops=flierp,
        whiskerprops=dict(linewidth=0.8),
        boxprops=dict(linewidth=0.8),
    )

    for box in boxplot['boxes']:
        box.set_facecolor(color)
        box.set_alpha(0.5)

    ax.set_title(vlabel, fontsize=13)
    ax.set_xticks(available_dims)
    ax.tick_params(axis='x', labelrotation=45, labelsize=10)
    ax.tick_params(axis='y', labelsize=10)
    ax.yaxis.grid(True, linewidth=0.3, alpha=0.5)
    ax.set_xlabel('Dimension $d$', fontsize=11)
    if index % ncols == 0:
        ax.set_ylabel(r'Mean step size $\alpha$', fontsize=11)

# Hide unused cells
for i in range(n, nrows * ncols):
    axs_flat[i].set_visible(False)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=9, label='mean'),
    Line2D([0], [0], color='black', lw=2, label='median'),
]
if has_legend_cell:
    fig.legend(handles=legend_handles, loc='lower right',
               bbox_to_anchor=(0.95, 0.12), frameon=True, fontsize=12,
               title='Legend', title_fontsize=11)
else:
    fig.legend(handles=legend_handles, loc='lower center',
               bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=True, fontsize=12,
               title='Legend', title_fontsize=11)

plt.suptitle('Mean Step Size over Dimensions', fontsize=15, y=1.01)
plt.tight_layout()

path = os.path.join(FIGURES_DIR, 'boxplot_alpha.pdf')
plt.savefig(path, dpi=300, bbox_inches='tight')
print(f'Saved:{path}')
plt.show()


## Tables - Regret Summary

In [ ]:
import csv

vlabels = [vl for _, vl in VARIANTS]
regret = {}
for index_dim, dim in enumerate(available_dims):
    fmax = np.array(f_max_corrected[dim])
    regret[dim] = {}
    for vname, vlabel in VARIANTS:
        best_at_end = interpolated[vname][index_dim].numpy()[:, -1]
        regret[dim][vlabel] = np.clip((fmax - best_at_end) / fmax, 0, None)

metrics = ['Mean', 'Std', 'Median']
col_header = ['Dim']
for vl in vlabels:
   for m in metrics:
      col_header.append(f'{vl}_{m}')
        
csv_path = os.path.join(TABLES_DIR, 'regret_summary.csv')
os.makedirs(TABLES_DIR, exist_ok=True)
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(col_header)
    for dim in available_dims:
        row = [dim]
        for vl in vlabels:
            entry = regret[dim][vl]
            row += [f'{np.mean(entry):.4f}', f'{np.std(entry):.4f}', f'{np.median(entry):.4f}']
        w.writerow(row)

print(f'Saved: {csv_path}')


## Tables - Step & Alpha Statistics

In [ ]:
import csv

# steps and alphas are loaded in function above
def aggregate(vals):
    # v == v false for naN values
    a = np.array([v for v in vals if v == v], dtype=float)
    if len(a) == 0:
        return [float('nan')] * 3
    return [float(np.mean(a)), float(np.std(a)), float(np.median(a))]

def format(v, dec=4):
    return '---' if (v != v) else f'{v:.{dec}f}'

vlabels2 = [vl for _, vl in VARIANTS]
metrics2  = ['Mean', 'Std', 'Med']

col_header2 = (['Dim']
             + [f'{vl}_Steps_{m}' for vl in vlabels2 for m in metrics2]
             +[f'{vl}_Alpha_{m}' for vl in vlabels2 for m in metrics2])

csv_path2 = os.path.join(TABLES_DIR, 'step_summary.csv')
os.makedirs(TABLES_DIR, exist_ok=True)
with open(csv_path2, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(col_header2)
    for dim in available_dims:
        row = [dim]
        for vname, _ in VARIANTS:
            row += [format(v, 1) for v in aggregate(step_totals[vname].get(dim, []))]
        for vname, _ in VARIANTS:
            row += [format(v) for v in aggregate(alpha_means[vname].get(dim, []))]
        w.writerow(row)

print(f'Saved: {csv_path2}')